# LOPOCV LightGBM + Optuna

## Purpose
Run leave-one-protein-out cross-validation (LOPOCV) with Optuna-based hyperparameter optimization for configured experiment settings.

## Input
- CSV table with all features and labels (output of the RMSD/labeling step)
- Feature lists from `features_global/*.csv` (output of the feature selection step)

## Output
- LOPOCV evaluation outputs and summary files used to build SI performance tables

## Run Order
1. Feature generation / selection notebooks (01--04)
2. This notebook
3. Feature importance analysis notebooks (06--08)

In [ ]:
# =====================
# Cell 1: Configuration
# =====================
# Update INPUT_CSV and OUT_DIR to match your local directory structure before running.
INPUT_CSV = "/path/to/input.csv"
OUT_DIR   = "/path/to/output"
TARGETS   = ["label_2p5", "label_5p0"]
PDB_COL   = "pdb_id"

EXPERIMENTS = [
    {"name": "ds",         "use_desc": False, "use_md": False},
    {"name": "ds_desc",    "use_desc": True,  "use_md": False},
    {"name": "ds_md",      "use_desc": False, "use_md": True},
    {"name": "ds_desc_md", "use_desc": True,  "use_md": True},
]

SEED          = 1    # Change this value for each run (1–50)
SEED_TAG      = f"seed_{SEED}"
# In the study, this notebook was executed 50 times with SEED = 1, 2, ..., 50
# to assess variability across random seeds.
N_TRIALS      = 400  # Number of Optuna trials; ~400 is a reasonable default
INNER_SPLITS  = 5
EARLY_STOP    = 20


In [ ]:
# =====================
# Cell 2: Libraries
# =====================
import os, json, warnings
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import lightgbm as lgb

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    balanced_accuracy_score, precision_recall_curve
)

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.dpi"] = 140
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
# =====================
# Cell 3: Utilities
# =====================
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def scale_pos_weight_from_y(y: np.ndarray) -> float:
    pos, neg = int((y==1).sum()), int((y==0).sum())
    return float(neg / max(pos, 1))

def metric_dict(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true))==2 else float("nan"),
        "pr_auc": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
    }

def best_f1_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    thresholds = np.r_[0.0, thr, 1.0]
    f1s = [f1_score(y_true, (y_prob >= t).astype(int), zero_division=0) for t in thresholds]
    i = int(np.argmax(f1s))
    return float(thresholds[i]), float(f1s[i])

def lgb_base_params() -> Dict:
    return {
        "device_type": "cpu",
        "num_threads": -1,      # use all available threads (-1 = auto)
        "tree_learner": "serial",
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "feature_pre_filter": False,
        "force_row_wise": True,
        "deterministic": True,
        "seed": SEED,
    }

# ---- Strategy B: load feature set from features_global ----
FEATURES_GLOBAL_DIR = Path(OUT_DIR) / "features_global"

def load_global_features(exp_name: str) -> List[str]:
    """
    Load feature names from features_global/{exp_name}.csv (column: 'feature').
    """
    feat_path = FEATURES_GLOBAL_DIR / f"{exp_name}.csv"
    if not feat_path.exists():
        raise FileNotFoundError(
            f"features_global not found: {feat_path}\n"
            "Please run the feature-selection notebook first to create features_global/*.csv."
        )
    feats = pd.read_csv(feat_path)["feature"].astype(str).tolist()
    if len(feats) == 0:
        raise ValueError(f"features_global is empty: {feat_path}")
    return feats


In [ ]:
# =====================
# Cell 4: Data loading
# =====================
df_raw = pd.read_csv(INPUT_CSV)

# Use pose rows only (exclude crystal structure rows)
df_raw = df_raw[df_raw["ligand_id"].astype(str).str.startswith("pose")].copy()
print(f"[Data] rows (pose only): {len(df_raw)}")


In [ ]:
# =====================
# Cell 5: Optuna objective function (inner CV)
# =====================
def make_objective(X, y, groups, inner_splits: int, early_stopping_rounds: int = EARLY_STOP):
    def objective(trial: optuna.Trial) -> float:
        params = {
            **lgb_base_params(),
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth":         trial.suggest_int("max_depth", 3, 10),
            "num_leaves":        trial.suggest_int("num_leaves", 15, 255),
            "feature_fraction":  trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction":  trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq":      trial.suggest_int("bagging_freq", 0, 5),
            "min_data_in_leaf":  trial.suggest_int("min_data_in_leaf", 10, 200),
            "lambda_l1":         trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2":         trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 2.0),
            "max_bin":           trial.suggest_int("max_bin", 128, 512),
        }

        n_estimators = trial.suggest_int("n_estimators", 100, 1200)

        gkf = GroupKFold(n_splits=min(inner_splits, len(np.unique(groups))))
        scores = []
        step = 0  # monotonically increasing counter for trial.report

        for tr_idx, va_idx in gkf.split(X, y, groups):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            params["scale_pos_weight"] = scale_pos_weight_from_y(y_tr)

            dtr = lgb.Dataset(X_tr, label=y_tr, free_raw_data=True)
            dva = lgb.Dataset(X_va, label=y_va, reference=dtr, free_raw_data=True)

            bst = lgb.train(
                params, dtr,
                num_boost_round=n_estimators,
                valid_sets=[dva],
                valid_names=["valid"],
                callbacks=[
                    lgb.early_stopping(early_stopping_rounds, verbose=False),
                    lgb.log_evaluation(period=0),
                ],
            )

            y_va_prob = bst.predict(X_va, num_iteration=bst.best_iteration)

            # skip single-class folds (AP is ill-defined)
            if len(np.unique(y_va)) < 2:
                continue

            scores.append(average_precision_score(y_va, y_va_prob))

            step += 1
            trial.report(float(np.mean(scores)), step=step)
            if trial.should_prune():
                raise optuna.TrialPruned()

        # if all folds are single-class, return chance-level AP ~ positive rate
        if len(scores) == 0:
            return float(np.mean(y))

        return float(np.mean(scores))

    return objective


In [ ]:
# =====================
# Cell 6: LOPOCV execution (training & evaluation only)
# =====================
def run_one_experiment(df_in: pd.DataFrame, exp_name: str, use_desc: bool, use_md: bool):
    print(f"\n=== Experiment: {exp_name}  (desc={use_desc}, md={use_md}) ===")

    # Strategy B: load fixed feature set from features_global
    global_feats = load_global_features(exp_name)
    X_all = df_in[global_feats].copy()

    root_dir = os.path.join(OUT_DIR, SEED_TAG, f"lgb_{exp_name}")
    ensure_dir(root_dir)

    summary_rows = []

    for target in TARGETS:
        print(f"\n===== Target: {target} ({exp_name}) =====")
        y = df_in[target].astype(int).values
        groups = df_in[PDB_COL].astype(str).values
        uniq_g = np.unique(groups)
        X = X_all.copy()

        out_dir = os.path.join(root_dir, f"{target}_lopocv")
        ensure_dir(out_dir)

        # Save feature list used in this run (= global_feats)
        pd.Series(X.columns, name="feature").to_csv(os.path.join(out_dir, "features.csv"), index=False)

        # OOF prediction buffer
        oof_prob = np.zeros(len(df_in))

        # LOPOCV outer loop
        for i, g in enumerate(uniq_g, 1):
            te_mask = (groups == g)
            tr_mask = ~te_mask
            X_tr, y_tr, grp_tr = X[tr_mask], y[tr_mask], groups[tr_mask]
            X_te, y_te = X[te_mask], y[te_mask]

            # inner tuning
            study = optuna.create_study(
                direction="maximize",
                sampler=optuna.samplers.TPESampler(seed=SEED),
                pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
            )
            objective = make_objective(X_tr, y_tr, grp_tr, INNER_SPLITS, EARLY_STOP)
            study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

            best = study.best_trial.params
            best_n = int(best.get("n_estimators", 500))
            params = {
                **lgb_base_params(),
                **{k: v for k, v in best.items() if k != "n_estimators"},
                "scale_pos_weight": scale_pos_weight_from_y(y_tr),
            }

            # Final training on all training folds
            bst = lgb.train(params, lgb.Dataset(X_tr, label=y_tr), num_boost_round=best_n)

            # Prediction
            prob = bst.predict(X_te, num_iteration=best_n)
            oof_prob[te_mask] = prob

            # Per-fold log
            try:
                test_auc = roc_auc_score(y_te, prob) if len(np.unique(y_te))==2 else float("nan")
            except ValueError:
                test_auc = float("nan")

            pr_auc = average_precision_score(y_te, prob)
            pos_cnt, n_all = int((y_te==1).sum()), len(y_te)
            print(f"PDB={g}  cv_ap={study.best_value:.4f}  test_auc={test_auc:.4f}  pr_auc={pr_auc:.4f}  pos={pos_cnt}/{n_all}")

            # Save fold best parameters
            with open(os.path.join(out_dir, f"fold_{i:02d}_{g}_best_params.json"), "w") as f:
                json.dump(best, f, indent=2)

            # --- Save materials for downstream SHAP / Permutation analysis ---
            fold_dir = os.path.join(out_dir, "materials", f"fold{i:02d}_{g}")
            ensure_dir(fold_dir)

            # Save model
            bst.save_model(os.path.join(fold_dir, "model.txt"), num_iteration=best_n)

            # Save test data and indices
            X_te.to_parquet(os.path.join(fold_dir, "X_te.parquet"))
            np.save(os.path.join(fold_dir, "y_te.npy"), y_te)
            np.save(os.path.join(fold_dir, "te_index.npy"), df_in.index[te_mask].to_numpy())
            pd.Series(X.columns, name="feature").to_csv(os.path.join(fold_dir, "features.csv"), index=False)

        # Save OOF predictions
        pd.DataFrame({"group": groups, "y_true": y, "y_prob": oof_prob}).to_csv(
            os.path.join(out_dir, "oof_predictions.csv"), index=False
        )

        # Metrics (threshold 0.5 and best F1)
        m05_auc = roc_auc_score(y, oof_prob) if len(np.unique(y))==2 else float("nan")
        m05_pr  = average_precision_score(y, oof_prob)
        thr_best, f1_best = best_f1_threshold(y, oof_prob)
        mbest = metric_dict(y, oof_prob, thr_best)

        metrics = {
            "n_samples": int(len(df_in)),
            "n_features": int(X.shape[1]),
            "n_groups": int(len(uniq_g)),

            # threshold-independent metrics
            "roc_auc": float(m05_auc),
            "pr_auc": float(m05_pr),

            # threshold-dependent metrics (best F1 on OOF)
            "bestF1": {
                "threshold": float(thr_best),
                "f1": float(mbest["f1"]),
                "balanced_acc": float(mbest["balanced_acc"]),
            },
        }
        with open(os.path.join(out_dir, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

        print(
            f"Done [{exp_name}]: {target}  "
            f"roc_auc={m05_auc:.4f}  pr_auc={m05_pr:.4f}  "
            f"F1@best={mbest['f1']:.4f} thr={thr_best:.3f}"
        )

        summary_rows.append({
            "experiment": exp_name,
            "target": target,
            "n_features": int(X.shape[1]),
            "roc_auc_oof": float(m05_auc),
            "pr_auc_oof": float(m05_pr),
            "bestF1": float(mbest["f1"]),
            "best_threshold": float(thr_best),
        })

    # Per-target summary
    pd.DataFrame(summary_rows).to_csv(os.path.join(root_dir, "summary_by_target.csv"), index=False)
    return summary_rows


In [ ]:
# =====================
# Cell 7: Combined execution & overall summary
# =====================
def run_all_experiments():
    all_rows = []
    for exp in EXPERIMENTS:
        rows = run_one_experiment(
            df_raw.copy(),
            exp_name=exp["name"],
            use_desc=exp["use_desc"],
            use_md=exp["use_md"],
        )
        all_rows.extend(rows)

    df_sum = pd.DataFrame(all_rows)
    seed_out_dir = os.path.join(OUT_DIR, SEED_TAG)
    Path(seed_out_dir).mkdir(parents=True, exist_ok=True)

    if not df_sum.empty:
        df_sum.to_csv(os.path.join(seed_out_dir, "summary_experiments.csv"), index=False)
        try:
            display(df_sum.sort_values(["target","pr_auc_oof"], ascending=[True, False]).reset_index(drop=True))
        except Exception:
            pass


In [ ]:
# =====================
# Cell 8: Execution
# =====================
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
run_all_experiments()
print("All experiments finished.")
